# 06 Build Route Database

## Purpose

This notebook builds a route-level service frequency table from METRO Houston GTFS data. The goal is to estimate how much scheduled service each route receives by counting the number of scheduled trips in `trips.txt`.

This step matters because ridership alone does not show whether a route is efficient or under pressure. A route with high ridership and relatively fewer scheduled trips may indicate stronger service productivity or potential overcrowding.

## Inputs

- `data/raw/metro_gtfs/merged/routes.txt`
- `data/raw/metro_gtfs/merged/trips.txt`

## Output

- `data/processed/route_frequency.csv`

The output file is used later to calculate riders per scheduled trip and the route investment priority score.


## 1. Import libraries and load GTFS route/trip data

The `routes.txt` file contains route names and identifiers. The `trips.txt` file contains scheduled trips for each route. Counting trips by route gives us a simple service supply metric.


In [ ]:
import pandas as pd

routes = pd.read_csv(
    "../data/raw/metro_gtfs/merged/routes.txt"
)

trips = pd.read_csv(
    "../data/raw/metro_gtfs/merged/trips.txt"
)

print("Routes shape:", routes.shape)
print("Trips shape:", trips.shape)

routes.head()


## 2. Count scheduled trips by route

Each row in `trips.txt` represents a scheduled trip. Grouping by `route_id` gives an estimate of how much service METRO schedules for each route in this GTFS feed.


In [ ]:
route_trip_counts = (
    trips.groupby("route_id")
    .size()
    .reset_index(name="scheduled_trips")
)

route_trip_counts.sort_values(
    "scheduled_trips",
    ascending=False
).head(10)


## 3. Join scheduled trips back to route names

This creates a readable route frequency table that includes route IDs, route names, and scheduled trip counts.


In [ ]:
master_routes = routes.merge(
    route_trip_counts,
    on="route_id",
    how="left"
)

master_routes[
    [
        "route_id",
        "route_short_name",
        "route_long_name",
        "scheduled_trips"
    ]
].sort_values(
    "scheduled_trips",
    ascending=False
).head(25)


## 4. Save processed route frequency data

The cleaned route frequency table is saved to `data/processed/` so later notebooks can merge service supply with ridership data.


In [ ]:
master_routes.to_csv(
    "../data/processed/route_frequency.csv",
    index=False
)

print("Saved route_frequency.csv to data/processed/")


## Summary

This notebook created a route-level service frequency dataset from GTFS scheduled trips. The resulting `route_frequency.csv` file allows later analysis to compare ridership demand against scheduled service supply.

This supports the project’s broader question: which Houston METRO corridors appear to be the strongest candidates for future transit investment?
